# Data Preprocessing & Feature Engineering

## Objective

The objective of this notebook is to prepare the retail sales dataset for machine learning model training.

Key tasks include:

- Data Cleaning
- Feature Engineering
- Categorical Encoding
- Train-Test Split
- Feature Selection
- Saving Processed Data

In [1]:
#importing libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("retail_sales_enriched.csv")
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Date,Year,Quarter,Month,WeekOfYear,DayOfWeek
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022-01-01,2022,1,1,52,5
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022-01-01,2022,1,1,52,5
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022-01-01,2022,1,1,52,5
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022-01-01,2022,1,1,52,5
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022-01-01,2022,1,1,52,5


In [3]:
print("Shape:", df.shape)
df.info()

Shape: (73100, 20)
<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Store ID            73100 non-null  str    
 1   Product ID          73100 non-null  str    
 2   Category            73100 non-null  str    
 3   Region              73100 non-null  str    
 4   Inventory Level     73100 non-null  int64  
 5   Units Sold          73100 non-null  int64  
 6   Units Ordered       73100 non-null  int64  
 7   Demand Forecast     73100 non-null  float64
 8   Price               73100 non-null  float64
 9   Discount            73100 non-null  int64  
 10  Weather Condition   73100 non-null  str    
 11  Holiday/Promotion   73100 non-null  int64  
 12  Competitor Pricing  73100 non-null  float64
 13  Seasonality         73100 non-null  str    
 14  Date                73100 non-null  str    
 15  Year                73100 non-null  int64  
 

## Feature Classification

Categorical Features:
- Store ID
- Product ID
- Category
- Region
- Weather Condition
- Seasonality

Numerical Features:
- Inventory Level
- Units Ordered
- Demand Forecast
- Price
- Discount
- Competitor Pricing

Target:
- Units Sold

In [18]:
target = "Units Sold"

X = df.drop(
    columns=[
        "Units Sold",
        "Date"
    ]
)

y = df[target]

In [19]:
categorical_cols = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

encoders = {}

for col in categorical_cols:
    
    le = LabelEncoder()
    
    X[col] = le.fit_transform(X[col])
    
    encoders[col] = le

In [20]:
X.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Quarter,Month,WeekOfYear,DayOfWeek
0,0,0,3,1,231,55,135.47,33.50,20,1,0,29.69,0,2022,1,1,52,5
1,0,1,4,2,204,66,144.04,63.01,20,3,0,66.16,0,2022,1,1,52,5
2,0,2,4,3,102,51,74.02,27.99,10,3,1,31.32,2,2022,1,1,52,5
3,0,3,4,1,469,164,62.18,32.72,10,0,1,34.74,0,2022,1,1,52,5
4,0,4,1,0,166,135,9.26,73.64,0,3,0,68.95,2,2022,1,1,52,5


Price Difference
Business Logic:If competitor price is lower,
sales may decrease.

In [21]:
X["Price_Difference"] = (
    X["Price"] -
    X["Competitor Pricing"]
)

In [22]:
X["Discount_Value"] = (
    X["Price"] *
    X["Discount"] / 100
)

In [23]:
X["Effective_Price"] = (
    X["Price"] -
    X["Discount_Value"]
)

In [24]:
X["Inventory_Coverage"] = (
    X["Inventory Level"] /
    (X["Demand Forecast"] + 1)
)

In [25]:
X.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,...,Seasonality,Year,Quarter,Month,WeekOfYear,DayOfWeek,Price_Difference,Discount_Value,Effective_Price,Inventory_Coverage
0,0,0,3,1,231,55,135.47,33.50,20,1,...,0,2022,1,1,52,5,3.81,6.700,26.800,1.692680
1,0,1,4,2,204,66,144.04,63.01,20,3,...,0,2022,1,1,52,5,-3.15,12.602,50.408,1.406509
2,0,2,4,3,102,51,74.02,27.99,10,3,...,2,2022,1,1,52,5,-3.33,2.799,25.191,1.359637
3,0,3,4,1,469,164,62.18,32.72,10,0,...,0,2022,1,1,52,5,-2.02,3.272,29.448,7.423235
4,0,4,1,0,166,135,9.26,73.64,0,3,...,2,2022,1,1,52,5,4.69,0.000,73.640,16.179337


In [26]:
print(X.columns.tolist())

['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Year', 'Quarter', 'Month', 'WeekOfYear', 'DayOfWeek', 'Price_Difference', 'Discount_Value', 'Effective_Price', 'Inventory_Coverage']


## Train-Test Split

80% Training Data

20% Testing Data

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    
    X,
    y,
    
    test_size=0.2,
    
    random_state=42
)

In [30]:
print("X Train:", X_train.shape)
print("X Test :", X_test.shape)

print("Y Train:", y_train.shape)
print("Y Test :", y_test.shape)

X Train: (58480, 22)
X Test : (14620, 22)
Y Train: (58480,)
Y Test : (14620,)


In [31]:
import joblib

joblib.dump(
    encoders,
    "label_encoders.pkl"
)

['label_encoders.pkl']

In [32]:
print(X_train.columns)

Index(['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level',
       'Units Ordered', 'Demand Forecast', 'Price', 'Discount',
       'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing',
       'Seasonality', 'Year', 'Quarter', 'Month', 'WeekOfYear', 'DayOfWeek',
       'Price_Difference', 'Discount_Value', 'Effective_Price',
       'Inventory_Coverage'],
      dtype='str')


In [33]:
joblib.dump(
    X_train,
    "X_train.pkl"
)

joblib.dump(
    X_test,
    "X_test.pkl"
)

joblib.dump(
    y_train,
    "y_train.pkl"
)

joblib.dump(
    y_test,
    "y_test.pkl"
)

['y_test.pkl']

# Preprocessing Summary

Completed:

✅ Encoded categorical variables

✅ Created business-oriented features

✅ Prepared feature matrix

✅ Split data into train and test sets

✅ Saved encoders for deployment


In [35]:
print(
    X_train.select_dtypes(include=["object"]).columns
)

Index([], dtype='str')


In [36]:
print("Date" in X_train.columns)

False
